# Initialize `oil_network` assignments — orchestrator

Thin orchestrator that runs every assignment notebook in sequence. Each child notebook owns its own slice of `oil_network.variable_assignments` (and `oil_network.timeseries` + `timeseries_data` for source-bound notebooks).

## Execution order matters

TS-bound notebooks must run **before** formula-bound notebooks, because formulas reference timeseries by id and may also reference variables that are TS-bound. The orchestrator enforces this ordering by listing source notebooks first.

## Per-notebook responsibility

| Notebook | Touches | Status |
|---|---|---|
| [assign_eia.ipynb](assign_eia.ipynb) | `oil_network.timeseries` (`source='eia'` rows) + `timeseries_data` + `variable_assignments` (TS-bound rows) | active |
| `assign_cer.ipynb` | future: per-pipeline cross-border flows from CER | future |
| [assign_formulas.ipynb](assign_formulas.ipynb) | `variable_assignments` (formula-bound rows): derived, zeros, latents | active |

Adding a new source assignment is the same 2-step pattern as the data loader: drop a new `assign_<source>.ipynb` next to this orchestrator, append to the `ASSIGNERS` list. Formula notebooks come last.

In [1]:
import subprocess
import sys
from datetime import datetime
from pathlib import Path

ASSIGNERS = [
    # 0. Structural defaults must exist BEFORE assign_formulas runs.
    "populate_node_type_defaults.py",
    "migrations/build_us_crude_grade_map.py",         # US crude grade registry + hierarchy

    # 1. TS-bound observations (must run before formula-bound).
    "assign_eia.ipynb",

    # 2. Formula-bound derivations + structural-zero overrides.
    "assign_formulas.ipynb",

    # 3. Aggregation graph + per-scenario node role tags.
    "migrations/add_aggregation_constituents.py",
    "migrations/add_node_roles.py",

    # 4. Post-load data migrations (chronological pass order).
    "migrations/repoint_foreign_supply_to_imports_agg.py",   # 8th pass
    "migrations/repoint_canadian_corridor_ts.py",            # 8th pass
    "migrations/add_padd2_canadian_imports_agg.py",          # 9th pass
    "migrations/add_partition_aggregates.py",                # 10th pass
    "migrations/split_bakken_gathering.py",                  # 11th pass
    "migrations/seventeenth_pass_xstate_membership.py",      # 17th pass
    "migrations/wire_bakken_xstate_into_inter_padds.py",     # post-17th: pipe_bakken_xstate constituent
    "migrations/eighteenth_pass_constraint_membership.py",   # 18th pass
    "migrations/twenty_first_pass_spr_under_padd3.py",
    "migrations/promote_spr_total_to_balance_role.py",
    "migrations/add_padd_stock_decomposition.py",
    "migrations/refactor_jones_act_into_inter_padd_agg.py",
    "migrations/add_canadian_pipelines_inventory_membership.py",   # Canadian cross-border pipes under PADD2

    # 5. View layer (dependency order).
    "create_v_effective_assignments.py",
    "migrations/twelfth_pass_cleanup.py",
    "migrations/sixteenth_pass_cleanup.py",
    "init_resolver_tables.py",
    "migrations/thirteenth_pass_views.py",
    "migrations/patch_v_aggregation_consistency_missing_data.py",

    # 5b. Capacity layer.
    "create_asset_capacities.py",
    "migrations/patch_pipeline_production_capacities.py",
    "migrations/patch_pipeline_timeline.py",
    "populate_asset_capacities.py",
    "create_variable_constraints.py",
    "create_v_effective_constraints.py",

    # 6. Resolver - populates scenario_resolved_values + auto-refreshes L4.
    "resolve_scenario.py",

    # 7. Post-resolution audits + balance-UI precomputed sums.
    "migrations/patch_production_caps_from_peaks.py",
    "populate_asset_capacities.py",
    "create_v_partition_sums.py",            # pre-computes balance-UI partition sums + intra
    "create_v_resolution_anomalies.py",      # data-quality view
    "audit_capacity_violations.py",
    "audit_resolution_anomalies.py",
]

TIMEOUT_SEC = 600


def run_step(step_path):
    if step_path.endswith(".ipynb"):
        cmd = [sys.executable, "-m", "jupyter", "nbconvert",
               "--to", "notebook", "--execute", "--inplace",
               f"--ExecutePreprocessor.timeout={TIMEOUT_SEC}", step_path]
    elif step_path.endswith(".py"):
        cmd = [sys.executable, step_path]
    else:
        raise ValueError(f"unknown step type: {step_path}")
    return subprocess.run(cmd, capture_output=True, text=True)


failures = []
for nb in ASSIGNERS:
    if not Path(nb).exists():
        print(f"  X {nb}: file missing")
        failures.append((nb, "file missing"))
        continue
    print(f"[{datetime.now():%H:%M:%S}] > Running {nb}...")
    res = run_step(nb)
    if res.returncode != 0:
        tail_lines = res.stderr.strip().splitlines()[-20:]
        failures.append((nb, tail_lines))
        print(f"[{datetime.now():%H:%M:%S}]   X FAILED (exit {res.returncode})")
    else:
        print(f"[{datetime.now():%H:%M:%S}]   OK done")

print()
if failures:
    print(f"{len(failures)} assigner(s) failed:")
    for nb, err in failures:
        print(f"--- {nb} ---")
        if isinstance(err, list):
            for ln in err: print(ln)
        else:
            print(err)
        print()
else:
    print(f"OK all {len(ASSIGNERS)} assigner(s) ran cleanly")


[14:24:44] > Running populate_node_type_defaults.py...


[14:24:44]   OK done
[14:24:44] > Running migrations/build_us_crude_grade_map.py...


[14:24:45]   OK done
[14:24:45] > Running assign_eia.ipynb...


[14:24:52]   X FAILED (exit 1)
[14:24:52] > Running assign_formulas.ipynb...


[14:25:00]   OK done
[14:25:00] > Running migrations/add_aggregation_constituents.py...


...

[14:25:06]   OK done
[14:25:06] > Running resolve_scenario.py...


[14:25:06]   X FAILED (exit 1)
[14:25:06] > Running migrations/patch_production_caps_from_peaks.py...
[14:25:06]   OK done
[14:25:06] > Running populate_asset_capacities.py...


[14:25:06]   OK done
[14:25:06] > Running create_v_partition_sums.py...


[14:25:07]   OK done
[14:25:07] > Running create_v_resolution_anomalies.py...
[14:25:07]   OK done
[14:25:07] > Running audit_capacity_violations.py...


[14:25:07]   OK done
[14:25:07] > Running audit_resolution_anomalies.py...
[14:25:07]   OK done

5 assigner(s) failed:
--- assign_eia.ipynb ---
ProgrammingError: (psycopg2.errors.UndefinedTable) relation "oil_network_data_loader.timeseries" does not exist
LINE 9:     FROM oil_network_data_loader.timeseries s
                 ^

[SQL: 
    INSERT INTO oil_network.timeseries_data
        (timeseries_id, observation_date, saved_date, value)
    SELECT
        %(ts_id)s,
        s.timeseries_date::date,
        s.timeseries_published_date::date,
        s.timeseries_value * %(scale)s
    FROM oil_network_data_loader.timeseries s
    WHERE s.source = 'eia'
      AND s.timeseries_id = %(source_id)s
    ON CONFLICT (timeseries_id, observation_date, saved_date) DO UPDATE
        SET value = EXCLUDED.value
]
[parameters: {'ts_id': 'eia:MCRFPNM2', 'scale': 1.0, 'source_id': 'MCRFPNM2'}]
(Background on this error at: https://sqlalche.me/e/20/f405)

--- migrations/add_partition_aggregates.py ---
T